In [334]:
import pandas as pd
import numpy as np

In [335]:
balls = pd.read_parquet("../ml-service/data/processed/v3_beta/clean_deliveries.parquet")
matches = pd.read_parquet("../ml-service/data/processed/v3_beta/clean_matches.parquet")

In [336]:
balls.head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs
0,335982,0,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,0,0,0,0.000000,1.000000,0.000000,Not Out,2008-04-18,1.000000
1,335982,0,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.000000,0.000000,0.000000,Not Out,2008-04-18,0.000000
2,335982,0,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,1,0,0.000000,0.000000,0.000000,Not Out,2008-04-18,1.000000
3,335982,0,0,4,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.000000,0.000000,0.000000,Not Out,2008-04-18,0.000000
4,335982,0,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.000000,0.000000,0.000000,Not Out,2008-04-18,0.000000


In [337]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs'],
      dtype='object')

In [338]:
balls.groupby(['matchId','inning','over','ball']).size().value_counts()

1    277161
2        32
Name: count, dtype: int64

In [339]:
duplicates = balls[
    balls.duplicated(['matchId','inning','over','ball'], keep=False)
].sort_values(['matchId','inning','over','ball'])
duplicates.head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs
5635,336005,1,6,1,Rajasthan Royals,Chennai Super Kings,GC Smith,SA Asnodkar,JA Morkel,0,0,0,0.000000,0.000000,0.000000,Not Out,2008-05-04,0.000000
5636,336005,1,6,1,Rajasthan Royals,Chennai Super Kings,GC Smith,SA Asnodkar,JA Morkel,0,0,0,0.000000,4.000000,0.000000,Not Out,2008-05-04,4.000000
9339,336024,0,18,1,Mumbai Indians,Deccan Chargers,PR Shah,YV Takawale,DP Vijaykumar,0,1,0,0.000000,0.000000,0.000000,Not Out,2008-05-18,1.000000
9340,336024,0,18,1,Mumbai Indians,Deccan Chargers,PR Shah,YV Takawale,DP Vijaykumar,1,0,0,0.000000,0.000000,0.000000,Not Out,2008-05-18,1.000000
35713,419141,1,1,1,Deccan Chargers,Rajasthan Royals,AC Gilchrist,VVS Laxman,SR Watson,0,1,0,0.000000,0.000000,0.000000,Not Out,2010-04-05,1.000000


In [340]:
len(duplicates)

64

In [341]:
balls = (
    balls.groupby(['matchId','inning','over','ball'], as_index=False)
    .agg({
        'batsman_runs': 'sum',
        'isWide': 'sum',
        'isNoBall': 'sum',
        'Byes': 'sum',
        'LegByes': 'sum',
        'Penalty': 'sum',
        'total_runs': 'sum',  # important

        'batting_team': 'first',
        'bowling_team': 'first',
        'batsman': 'first',
        'non_striker': 'first',
        'bowler': 'first',
        'player_dismissed': lambda x: x[x != "Not Out"].iloc[0] if any(x != "Not Out") else "Not Out",
        'date': 'first'
    })
)

In [342]:
balls.groupby(['matchId','inning','over','ball']).size().value_counts()

1    277193
Name: count, dtype: int64

In [343]:
balls = balls.sort_values(['matchId', 'inning', 'over', 'ball']).reset_index(drop=True)

In [344]:
balls.groupby(['matchId','inning','over','ball']).size().value_counts()

1    277193
Name: count, dtype: int64

In [345]:
mask = balls['isNoBall'] > 1

balls.loc[mask, 'batsman_runs'] += (balls.loc[mask, 'isNoBall'] - 1)
balls.loc[mask, 'isNoBall'] = 1

In [346]:
balls['repeat'] = np.where(balls['isWide'] > 0, balls['isWide'], 1)

balls = balls.loc[balls.index.repeat(balls['repeat'])].copy()

balls.loc[balls['isWide'] > 0, 'isWide'] = 1

balls.drop(columns=['repeat'], inplace=True)

In [347]:
balls['is_legal'] = ((balls['isWide'] == 0) & (balls['isNoBall'] == 0)).astype(int)

balls['ball'] = (
    balls.groupby(['matchId','inning','over'])['is_legal']
    .cumsum()
)

balls['ball'] = balls['ball'].replace(0, np.nan)
balls['ball'] = (
    balls.groupby(['matchId','inning','over'])['ball']
    .ffill()
    .fillna(1)
)

In [348]:
balls = balls.reset_index(drop=True)

In [349]:
balls['isWide'].unique()

array([0, 1])

In [350]:
balls['ball'].unique()

array([1., 2., 3., 4., 5., 6., 7.])

In [351]:
balls = balls[balls['ball'] <= 6]

In [352]:
balls['ball'].unique()

array([1., 2., 3., 4., 5., 6.])

In [353]:
balls.groupby(['matchId','inning','over'])['is_legal'].sum().max()

np.int64(6)

In [354]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal'],
      dtype='object')

In [355]:
balls['legal_ball'] = (balls['isWide'] == 0) & (balls['isNoBall'] == 0)

balls['balls_bowled'] = balls.groupby(['matchId', 'inning'])['legal_ball'].cumsum()

balls['balls_remaining'] = 120 - balls['balls_bowled']

balls.drop(columns=['balls_bowled'], inplace=True)

In [356]:
balls.groupby(['matchId','inning'])['legal_ball'].sum().max()

np.int64(120)

In [357]:
balls = balls.reset_index(drop=True)

In [358]:
balls['balls_remaining'].unique()

array([119, 118, 117, 116, 115, 114, 113, 112, 111, 110, 109, 108, 107,
       106, 105, 104, 103, 102, 101, 100,  99,  98,  97,  96,  95,  94,
        93,  92,  91,  90,  89,  88,  87,  86,  85,  84,  83,  82,  81,
        80,  79,  78,  77,  76,  75,  74,  73,  72,  71,  70,  69,  68,
        67,  66,  65,  64,  63,  62,  61,  60,  59,  58,  57,  56,  55,
        54,  53,  52,  51,  50,  49,  48,  47,  46,  45,  44,  43,  42,
        41,  40,  39,  38,  37,  36,  35,  34,  33,  32,  31,  30,  29,
        28,  27,  26,  25,  24,  23,  22,  21,  20,  19,  18,  17,  16,
        15,  14,  13,  12,  11,  10,   9,   8,   7,   6,   5,   4,   3,
         2,   1,   0, 120])

In [359]:
balls['legal_ball'][balls['balls_remaining']==120].unique()

array([False])

In [360]:
balls['over_number'] = balls['over'].astype(int) + 1
balls['phase'] = np.select(
    [
        balls['over_number'] <= 6,
        balls['over_number'] <= 15
    ],
    [0, 1],
    default=2
)
# balls.drop(columns=['over_number'], inplace=True)

In [361]:
balls['over_number'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20])

In [362]:
balls[balls['phase']==1].head(5)

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,total_runs,batting_team,bowling_team,batsman,non_striker,bowler,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase
42,335982,0,6,1.000000,1,0,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,RT Ponting,AA Noffke,Not Out,2008-04-18,1,True,83,7,1
43,335982,0,6,2.000000,1,0,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,RT Ponting,BB McCullum,AA Noffke,Not Out,2008-04-18,1,True,82,7,1
44,335982,0,6,3.000000,1,0,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,RT Ponting,AA Noffke,Not Out,2008-04-18,1,True,81,7,1
45,335982,0,6,4.000000,2,0,0,0.000000,0.000000,0.000000,2.000000,Kolkata Knight Riders,Royal Challengers Bangalore,RT Ponting,BB McCullum,AA Noffke,Not Out,2008-04-18,1,True,80,7,1
46,335982,0,6,5.000000,1,0,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,RT Ponting,BB McCullum,AA Noffke,Not Out,2008-04-18,1,True,79,7,1


In [363]:
balls["total_runs"] = (balls["batsman_runs"]+ balls["isWide"]+ balls["isNoBall"]+ balls["Byes"]+ balls["LegByes"]+ balls["Penalty"])
balls['current_score'] = balls.groupby(['matchId', 'inning'])['total_runs'].cumsum()
# balls.drop(columns=['total_runs'], inplace=True)

In [364]:
balls[['over', 'ball','current_score','isWide','isNoBall']].head(10)

,over,ball,current_score,isWide,isNoBall
0,0,1.000000,1.000000,0,0
1,0,2.000000,1.000000,0,0
2,0,2.000000,2.000000,1,0
3,0,3.000000,2.000000,0,0
4,0,4.000000,2.000000,0,0
5,0,5.000000,2.000000,0,0
6,0,6.000000,3.000000,0,0
7,1,1.000000,3.000000,0,0
8,1,2.000000,7.000000,0,0
9,1,3.000000,11.000000,0,0


In [365]:
balls['current_score'].unique().max()

np.float64(287.0)

In [366]:
balls.loc[balls['Penalty'] == 5, 'batsman_runs'] = 5

In [367]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal', 'legal_ball', 'balls_remaining', 'over_number',
       'phase', 'current_score'],
      dtype='object')

In [368]:
balls["batsman_runs"] = balls["batsman_runs"] + balls["Byes"] + balls["LegByes"]
balls['batsman_runs'].unique()

array([1., 0., 4., 6., 2., 5., 3., 7.])

In [369]:
balls.loc[balls['batsman_runs'] > 6, 'batsman_runs'] = 6


In [370]:
balls = balls.reset_index(drop=True)

In [371]:
balls['player_dismissed'].head(2)

0    Not Out
1    Not Out
Name: player_dismissed, dtype: object

In [372]:
len(balls['player_dismissed'].unique())

657

In [373]:
balls['is_wicket'] = (balls['player_dismissed'] != 'Not Out').astype(int)
balls['wickets_fallen'] = balls.groupby(['matchId', 'inning'])['is_wicket'].cumsum()
balls.drop(columns=['is_wicket'], inplace=True)

In [374]:
balls['wickets_fallen'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [375]:
balls.head()

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,total_runs,batting_team,bowling_team,batsman,non_striker,bowler,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen
0,335982,0,0,1.000000,1.000000,0,0,0.000000,1.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,Not Out,2008-04-18,1,True,119,1,0,1.000000,0
1,335982,0,0,2.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,Not Out,2008-04-18,1,True,118,1,0,1.000000,0
2,335982,0,0,2.000000,0.000000,1,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,Not Out,2008-04-18,0,False,118,1,0,2.000000,0
3,335982,0,0,3.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,Not Out,2008-04-18,1,True,117,1,0,2.000000,0
4,335982,0,0,4.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,Not Out,2008-04-18,1,True,116,1,0,2.000000,0


In [376]:
balls = balls.reset_index(drop=True)

In [377]:
first_innings_score = (
    balls[balls['inning'] == 0]
    .groupby('matchId')['current_score']
    .max()
)
balls['target'] = balls['matchId'].map(first_innings_score)
balls.loc[balls['inning'] == 1, 'target'] = balls['target'] + 1
balls.loc[balls['inning'] == 0, 'target'] = 0

In [378]:
balls[balls['target']>0].head(2)

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,total_runs,batting_team,bowling_team,batsman,non_striker,bowler,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen,target
129,335982,1,0,1.000000,1.000000,0,0,0.000000,0.000000,0.000000,1.000000,Royal Challengers Bangalore,Kolkata Knight Riders,R Dravid,W Jaffer,AB Dinda,Not Out,2008-04-18,1,True,119,1,0,1.000000,0,223.000000
130,335982,1,0,1.000000,0.000000,1,0,0.000000,0.000000,0.000000,1.000000,Royal Challengers Bangalore,Kolkata Knight Riders,W Jaffer,R Dravid,AB Dinda,Not Out,2008-04-18,0,False,119,1,0,2.000000,0,223.000000


In [379]:
balls['target'].unique().max()

np.float64(288.0)

In [380]:
over_runs = (
    balls.groupby(['matchId', 'inning', 'over_number'])['total_runs']
    .sum()
    .reset_index(name='over_runs')
)
over_runs['last_over_runs'] = (
    over_runs.groupby(['matchId', 'inning'])['over_runs']
    .shift(1)
)
balls = balls.merge(
    over_runs[['matchId', 'inning', 'over_number', 'last_over_runs']],
    on=['matchId', 'inning', 'over_number'],
    how='left'
)
balls['last_over_runs'] = balls['last_over_runs'].fillna(0)
balls['last_over_runs'] = balls['last_over_runs'].astype(int)

In [381]:
balls[['over_number', 'total_runs', 'last_over_runs']].head(10)

,over_number,total_runs,last_over_runs
0,1,1.000000,0
1,1,0.000000,0
2,1,1.000000,0
3,1,0.000000,0
4,1,0.000000,0
5,1,0.000000,0
6,1,1.000000,0
7,2,0.000000,3
8,2,4.000000,3
9,2,4.000000,3


In [382]:
balls['last_over_runs'].unique()

array([ 0,  3, 18,  6, 23, 10,  1,  7,  5,  4, 15, 12, 24, 14, 21,  8,  2,
       11,  9, 13, 19, 20, 17, 16, 26, 30, 22, 27, 25, 28, 33, 37, 31, 29,
       32])

In [383]:
len(balls['batsman_runs'][balls['batsman_runs']==5])

76

In [384]:
balls['batsman_runs'].unique()

array([1., 0., 4., 6., 2., 5., 3.])

In [385]:
balls["total_balls_in_over"] = balls.groupby(
    ["matchId", "inning", "over"]
).cumcount() + 1

In [386]:
balls.groupby(['matchId','inning','over','ball','total_balls_in_over']).size().value_counts()

1    278989
Name: count, dtype: int64

In [387]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal', 'legal_ball', 'balls_remaining', 'over_number',
       'phase', 'current_score', 'wickets_fallen', 'target', 'last_over_runs',
       'total_balls_in_over'],
      dtype='object')

In [388]:
balls['total_balls_in_over'].unique().max()

np.int64(17)

In [389]:
balls.head()

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,total_runs,batting_team,bowling_team,batsman,non_striker,bowler,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen,target,last_over_runs,total_balls_in_over
0,335982,0,0,1.000000,1.000000,0,0,0.000000,1.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,Not Out,2008-04-18,1,True,119,1,0,1.000000,0,0.000000,0,1
1,335982,0,0,2.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,Not Out,2008-04-18,1,True,118,1,0,1.000000,0,0.000000,0,2
2,335982,0,0,2.000000,0.000000,1,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,Not Out,2008-04-18,0,False,118,1,0,2.000000,0,0.000000,0,3
3,335982,0,0,3.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,Not Out,2008-04-18,1,True,117,1,0,2.000000,0,0.000000,0,4
4,335982,0,0,4.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,Not Out,2008-04-18,1,True,116,1,0,2.000000,0,0.000000,0,5


In [390]:
balls = balls.sort_values(
    ["matchId", "inning", "over", "total_balls_in_over"]
).reset_index(drop=True)

In [391]:
balls["is_boundary"] = balls["batsman_runs"].isin([4, 6]).astype(int)

def compute_balls_since_boundary(x):
    # cumulative number of boundaries seen so far
    groups = x.cumsum()

    # count balls since last boundary
    result = x.groupby(groups).cumcount()

    # from start of innings before first boundary:
    # 0,1,2,3...
    result[groups == 0] = range((groups == 0).sum())

    return result

balls["balls_since_boundary"] = (
    balls.groupby(["matchId", "inning"])["is_boundary"]
         .transform(compute_balls_since_boundary)
)

In [392]:
balls['balls_since_boundary'].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100])

In [393]:
len(balls[(balls['balls_since_boundary']>1) & (balls['over'] == 0) & (balls['total_balls_in_over'] == 1)])

0

In [394]:
# should only be {-1, 0}
balls[(balls['over'] == 0) & (balls['total_balls_in_over'] == 1)]['balls_since_boundary'].unique()

array([0])

In [395]:
balls[balls['is_boundary'] == 1]['balls_since_boundary'].unique()

array([0])

In [396]:
balls[(balls['over'] == 0) & (balls['ball'] == 1)][
    ['is_boundary', 'balls_since_boundary']
]

,is_boundary,balls_since_boundary
0,0,0
129,0,0
130,0,1
231,0,0
355,1,0
...,...,...
278488,0,0
278489,0,1
278617,1,0
278740,0,0


In [397]:
balls[(balls['over']==0) & (balls['ball']==1)].head(20)

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,total_runs,batting_team,bowling_team,batsman,non_striker,bowler,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen,target,last_over_runs,total_balls_in_over,is_boundary,balls_since_boundary
0,335982,0,0,1.000000,1.000000,0,0,0.000000,1.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,Not Out,2008-04-18,1,True,119,1,0,1.000000,0,0.000000,0,1,0,0
129,335982,1,0,1.000000,1.000000,0,0,0.000000,0.000000,0.000000,1.000000,Royal Challengers Bangalore,Kolkata Knight Riders,R Dravid,W Jaffer,AB Dinda,Not Out,2008-04-18,1,True,119,1,0,1.000000,0,223.000000,0,1,0,0
130,335982,1,0,1.000000,0.000000,1,0,0.000000,0.000000,0.000000,1.000000,Royal Challengers Bangalore,Kolkata Knight Riders,W Jaffer,R Dravid,AB Dinda,Not Out,2008-04-18,0,False,119,1,0,2.000000,0,223.000000,0,2,0,1
231,335983,0,0,1.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Chennai Super Kings,Kings XI Punjab,PA Patel,ML Hayden,B Lee,Not Out,2008-04-19,1,True,119,1,0,0.000000,0,0.000000,0,1,0,0
355,335983,1,0,1.000000,4.000000,0,0,0.000000,0.000000,0.000000,4.000000,Kings XI Punjab,Chennai Super Kings,K Goel,JR Hopes,JDP Oram,Not Out,2008-04-19,1,True,119,1,0,4.000000,0,241.000000,0,1,1,0
480,335984,0,0,1.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Rajasthan Royals,Delhi Daredevils,T Kohli,YK Pathan,GD McGrath,Not Out,2008-04-19,1,True,119,1,0,0.000000,0,0.000000,0,1,0,0
603,335984,1,0,1.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Delhi Daredevils,Rajasthan Royals,G Gambhir,V Sehwag,MM Patel,Not Out,2008-04-19,1,True,119,1,0,0.000000,0,130.000000,0,1,0,0
704,335985,0,0,1.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Mumbai Indians,Royal Challengers Bangalore,L Ronchi,ST Jayasuriya,P Kumar,Not Out,2008-04-20,1,True,119,1,0,0.000000,0,0.000000,0,1,0,0
827,335985,1,0,1.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,Royal Challengers Bangalore,Mumbai Indians,S Chanderpaul,R Dravid,A Nehra,Not Out,2008-04-20,1,True,119,1,0,0.000000,0,166.000000,0,1,0,0
950,335986,0,0,1.000000,1.000000,0,0,0.000000,0.000000,0.000000,1.000000,Deccan Chargers,Kolkata Knight Riders,AC Gilchrist,Y Venugopal Rao,AB Dinda,Not Out,2008-04-20,1,True,119,1,0,1.000000,0,0.000000,0,1,0,0


In [398]:
balls['percentage_target_achieved'] = np.where(
    balls['inning'] == 0,
    0.0,
    balls['current_score'] / balls['target']
)
balls['percentage_target_achieved'] = (
    balls['percentage_target_achieved']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


In [399]:
balls['percentage_target_achieved'][balls['inning']==1].head(2)

129   0.004484
130   0.008969
Name: percentage_target_achieved, dtype: float64

In [400]:
matches.columns

Index(['season', 'venue', 'winner_runs', 'date', 'winner', 'team1', 'team2',
       'winner_wickets', 'player_of_match', 'matchId'],
      dtype='object')

In [401]:
balls = balls.merge(
    matches[['matchId', 'venue']],
    on='matchId',
    how='left'
)

In [402]:
balls['venue'].head(240)

0                           M Chinnaswamy Stadium
1                           M Chinnaswamy Stadium
2                           M Chinnaswamy Stadium
3                           M Chinnaswamy Stadium
4                           M Chinnaswamy Stadium
                          ...                    
235    Punjab Cricket Association Stadium, Mohali
236    Punjab Cricket Association Stadium, Mohali
237    Punjab Cricket Association Stadium, Mohali
238    Punjab Cricket Association Stadium, Mohali
239    Punjab Cricket Association Stadium, Mohali
Name: venue, Length: 240, dtype: object

In [403]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal', 'legal_ball', 'balls_remaining', 'over_number',
       'phase', 'current_score', 'wickets_fallen', 'target', 'last_over_runs',
       'total_balls_in_over', 'is_boundary', 'balls_since_boundary',
       'percentage_target_achieved', 'venue'],
      dtype='object')

In [404]:
balls['balls_bowled'] = (
    balls.groupby(['matchId','inning'])['legal_ball']
    .cumsum()
)

balls['overs_bowled'] = balls['balls_bowled'] / 6

balls['current_run_rate'] = np.where(
    balls['balls_bowled'] > 0,
    balls['current_score'] / balls['overs_bowled'],
    0
)

In [405]:
balls['runs_required'] = balls['target'] - balls['current_score']

balls['overs_remaining'] = balls['balls_remaining'] / 6

balls['required_run_rate'] = (
    balls['runs_required'] / balls['overs_remaining']
)

# handle divide by 0
balls['required_run_rate'] = balls['required_run_rate'].replace([np.inf, -np.inf], 0)
balls['required_run_rate'] = balls['required_run_rate'].fillna(0)

balls.loc[balls['inning'] == 0, 'required_run_rate'] = 0
balls.loc[balls['runs_required'] <= 0, 'required_run_rate'] = 0

In [406]:
balls[['current_run_rate','required_run_rate']].describe()

,current_run_rate,required_run_rate
count,278989.000000,278989.000000
mean,7.664829,5.207731
std,2.443143,10.709678
min,0.000000,0.000000
25%,6.391304,0.000000
50%,7.659574,0.000000
75%,8.933333,8.933333
max,66.000000,792.000000


In [407]:
pd.set_option("display.max_columns", None)
balls[balls['current_run_rate']>42].head()

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,total_runs,batting_team,bowling_team,batsman,non_striker,bowler,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen,target,last_over_runs,total_balls_in_over,is_boundary,balls_since_boundary,percentage_target_achieved,venue,balls_bowled,overs_bowled,current_run_rate,runs_required,overs_remaining,required_run_rate
47406,501222,0,0,1.000000,0.000000,1,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,JH Kallis,BJ Haddin,Z Khan,Not Out,2011-04-22,0,False,119,1,0,8.000000,0,0.000000,0,5,0,4,0.000000,Eden Gardens,1,0.166667,48.000000,-8.000000,19.833333,0.000000
47407,501222,0,0,1.000000,0.000000,1,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,JH Kallis,BJ Haddin,Z Khan,Not Out,2011-04-22,0,False,119,1,0,9.000000,0,0.000000,0,6,0,5,0.000000,Eden Gardens,1,0.166667,54.000000,-9.000000,19.833333,0.000000
47408,501222,0,0,1.000000,0.000000,1,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,JH Kallis,BJ Haddin,Z Khan,Not Out,2011-04-22,0,False,119,1,0,10.000000,0,0.000000,0,7,0,6,0.000000,Eden Gardens,1,0.166667,60.000000,-10.000000,19.833333,0.000000
47409,501222,0,0,1.000000,0.000000,1,0,0.000000,0.000000,0.000000,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,JH Kallis,BJ Haddin,Z Khan,Not Out,2011-04-22,0,False,119,1,0,11.000000,0,0.000000,0,8,0,7,0.000000,Eden Gardens,1,0.166667,66.000000,-11.000000,19.833333,0.000000
85718,598034,1,0,1.000000,4.000000,0,0,4.000000,0.000000,0.000000,4.000000,Kolkata Knight Riders,Chennai Super Kings,MS Bisla,G Gambhir,DP Nannes,Not Out,2013-04-28,1,True,119,1,0,10.000000,0,201.000000,0,7,1,0,0.049751,"MA Chidambaram Stadium, Chepauk",1,0.166667,60.000000,191.000000,19.833333,9.630252


In [408]:
balls['over'] = balls['over']/20
np.set_printoptions(suppress=True)
balls['over'].unique()

array([0.  , 0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 ,
       0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

In [409]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal', 'legal_ball', 'balls_remaining', 'over_number',
       'phase', 'current_score', 'wickets_fallen', 'target', 'last_over_runs',
       'total_balls_in_over', 'is_boundary', 'balls_since_boundary',
       'percentage_target_achieved', 'venue', 'balls_bowled', 'overs_bowled',
       'current_run_rate', 'runs_required', 'overs_remaining',
       'required_run_rate'],
      dtype='object')

In [410]:
season_map = matches.set_index('matchId')['season']
balls['season'] = balls['matchId'].map(season_map)

In [411]:
# create cyclic features
balls['sin_ball'] = np.sin(2 * np.pi * balls['ball'] / 6)
balls['cos_ball'] = np.cos(2 * np.pi * balls['ball'] / 6)

In [412]:
pd.set_option('display.float_format', '{:.6f}'.format)
balls[['ball','sin_ball','cos_ball','isWide','batsman_runs']].head(10)

,ball,sin_ball,cos_ball,isWide,batsman_runs
0,1.000000,0.866025,0.500000,0,1.000000
1,2.000000,0.866025,-0.500000,0,0.000000
2,2.000000,0.866025,-0.500000,1,0.000000
3,3.000000,0.000000,-1.000000,0,0.000000
4,4.000000,-0.866025,-0.500000,0,0.000000
5,5.000000,-0.866025,0.500000,0,0.000000
6,6.000000,-0.000000,1.000000,0,1.000000
7,1.000000,0.866025,0.500000,0,0.000000
8,2.000000,0.866025,-0.500000,0,4.000000
9,3.000000,0.000000,-1.000000,0,4.000000


In [413]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal', 'legal_ball', 'balls_remaining', 'over_number',
       'phase', 'current_score', 'wickets_fallen', 'target', 'last_over_runs',
       'total_balls_in_over', 'is_boundary', 'balls_since_boundary',
       'percentage_target_achieved', 'venue', 'balls_bowled', 'overs_bowled',
       'current_run_rate', 'runs_required', 'overs_remaining',
       'required_run_rate', 'season', 'sin_ball', 'cos_ball'],
      dtype='object')

In [414]:
balls.drop(columns=['ball','batting_team', 'bowling_team','Byes', 'LegByes', 'Penalty','over_number','is_legal', 'legal_ball','is_boundary','legal_ball','balls_bowled', 'overs_bowled','runs_required', 'overs_remaining','date'], inplace=True)
balls.head()

,matchId,inning,over,batsman_runs,isWide,isNoBall,total_runs,batsman,non_striker,bowler,player_dismissed,balls_remaining,phase,current_score,wickets_fallen,target,last_over_runs,total_balls_in_over,balls_since_boundary,percentage_target_achieved,venue,current_run_rate,required_run_rate,season,sin_ball,cos_ball
0,335982,0,0.000000,1.000000,0,0,1.000000,SC Ganguly,BB McCullum,P Kumar,Not Out,119,0,1.000000,0,0.000000,0,1,0,0.000000,M Chinnaswamy Stadium,6.000000,0.000000,2008,0.866025,0.500000
1,335982,0,0.000000,0.000000,0,0,0.000000,BB McCullum,SC Ganguly,P Kumar,Not Out,118,0,1.000000,0,0.000000,0,2,1,0.000000,M Chinnaswamy Stadium,3.000000,0.000000,2008,0.866025,-0.500000
2,335982,0,0.000000,0.000000,1,0,1.000000,BB McCullum,SC Ganguly,P Kumar,Not Out,118,0,2.000000,0,0.000000,0,3,2,0.000000,M Chinnaswamy Stadium,6.000000,0.000000,2008,0.866025,-0.500000
3,335982,0,0.000000,0.000000,0,0,0.000000,BB McCullum,SC Ganguly,P Kumar,Not Out,117,0,2.000000,0,0.000000,0,4,3,0.000000,M Chinnaswamy Stadium,4.000000,0.000000,2008,0.000000,-1.000000
4,335982,0,0.000000,0.000000,0,0,0.000000,BB McCullum,SC Ganguly,P Kumar,Not Out,116,0,2.000000,0,0.000000,0,5,4,0.000000,M Chinnaswamy Stadium,3.000000,0.000000,2008,-0.866025,-0.500000


In [415]:
balls.columns

Index(['matchId', 'inning', 'over', 'batsman_runs', 'isWide', 'isNoBall',
       'total_runs', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'balls_remaining', 'phase', 'current_score', 'wickets_fallen', 'target',
       'last_over_runs', 'total_balls_in_over', 'balls_since_boundary',
       'percentage_target_achieved', 'venue', 'current_run_rate',
       'required_run_rate', 'season', 'sin_ball', 'cos_ball'],
      dtype='object')

In [416]:
balls['batsman_runs'] = balls['batsman_runs']/6
balls['total_runs'] = balls['total_runs']/6
balls['balls_remaining'] = balls['balls_remaining']/120
balls['wickets_fallen'] = balls['wickets_fallen']/10
balls['balls_since_boundary'] = balls['balls_since_boundary']/120

In [417]:
balls.head()

,matchId,inning,over,batsman_runs,isWide,isNoBall,total_runs,batsman,non_striker,bowler,player_dismissed,balls_remaining,phase,current_score,wickets_fallen,target,last_over_runs,total_balls_in_over,balls_since_boundary,percentage_target_achieved,venue,current_run_rate,required_run_rate,season,sin_ball,cos_ball
0,335982,0,0.000000,0.166667,0,0,0.166667,SC Ganguly,BB McCullum,P Kumar,Not Out,0.991667,0,1.000000,0.000000,0.000000,0,1,0.000000,0.000000,M Chinnaswamy Stadium,6.000000,0.000000,2008,0.866025,0.500000
1,335982,0,0.000000,0.000000,0,0,0.000000,BB McCullum,SC Ganguly,P Kumar,Not Out,0.983333,0,1.000000,0.000000,0.000000,0,2,0.008333,0.000000,M Chinnaswamy Stadium,3.000000,0.000000,2008,0.866025,-0.500000
2,335982,0,0.000000,0.000000,1,0,0.166667,BB McCullum,SC Ganguly,P Kumar,Not Out,0.983333,0,2.000000,0.000000,0.000000,0,3,0.016667,0.000000,M Chinnaswamy Stadium,6.000000,0.000000,2008,0.866025,-0.500000
3,335982,0,0.000000,0.000000,0,0,0.000000,BB McCullum,SC Ganguly,P Kumar,Not Out,0.975000,0,2.000000,0.000000,0.000000,0,4,0.025000,0.000000,M Chinnaswamy Stadium,4.000000,0.000000,2008,0.000000,-1.000000
4,335982,0,0.000000,0.000000,0,0,0.000000,BB McCullum,SC Ganguly,P Kumar,Not Out,0.966667,0,2.000000,0.000000,0.000000,0,5,0.033333,0.000000,M Chinnaswamy Stadium,3.000000,0.000000,2008,-0.866025,-0.500000


In [418]:
balls['current_score'] = balls['current_score']/200
balls['target'] = balls['target']/200
balls['last_over_runs'] = balls['last_over_runs']/200
balls['total_runs'] = balls['total_runs']/6
balls['total_balls_in_over'] = balls['total_balls_in_over']/10

In [419]:
balls.to_csv('data3.csv')